# 08.03 msame 推理验证与排错

本分册使用 msame 读取 OM、组织输入 bin、比较输出并按阶段定位转换和推理问题。


## 实验原理

ATC 生成 OM 后，可用 msame 在昇腾设备上运行模型，并与 ONNX Runtime 的输出比较。

| 工具 | 用途 |
| --- | --- |
| ATC | 模型转换 |
| msame | 单次或多次推理、输出读取 |
| Profiling | 性能瓶颈分析 |
| benchmark | 标准模型性能基准 |

msame 源码位于 Gitee 的 `ascend/tools` 仓库。编译前配置 CANN 环境，按仓库 README 编译后，产物通常位于 `out/msame`。


## 实验流程

### 1. 查看参数并运行最小示例

先通过 `--help` 核对本机 msame 的参数。不同版本的参数名称和选项可能不同。


In [ ]:
!/home/ma-user/work/tools/msame/out/msame --help

静态 shape 场景常用四个参数：

| 参数 | 含义 |
| --- | --- |
| `--model` | OM 模型路径 |
| `--input` | 输入数据，单个 bin 文件或目录 |
| `--output` | 输出目录，msame 在其下建时间戳子目录 |
| `--outfmt` | 输出格式，`BIN` 或 `TXT` |


In [ ]:
# %%bash
# source /usr/local/Ascend/ascend-toolkit/set_env.sh

!/home/ma-user/work/tools/msame/out/msame --model l08_workspace/demo_model.om \
      --input l08_workspace/input.bin \
      --output l08_workspace/msame_out \
      --outfmt BIN

省略 `--input` 时，msame 会使用自动生成的数据推理，适合观察耗时。精度比较应使用与业务相符的输入数据。


### 2. 准备输入并读取输出

#### 组织 bin 文件

bin 是裸二进制，不含 shape 和 dtype。单输入使用一个文件；多输入时，msame 按文件排序后的顺序匹配模型输入，因此文件名排序应与模型输入顺序一致。

可使用数字前缀固定顺序：


In [ ]:
import numpy as np, os

os.makedirs("l08_workspace/multi_in", exist_ok=True)
np.random.seed(0)

inputs = {
    "00_input_ids": (np.random.randint(0, 1000, (1, 128)).astype(np.int64), np.int64),
    "01_attention_mask": (np.ones((1, 128), dtype=np.int64), np.int64),
    "02_token_type_ids": (np.zeros((1, 128), dtype=np.int64), np.int64),
}

for name, (arr, dt) in inputs.items():
    path = "l08_workspace/multi_in/%s.bin" % name
    arr.tofile(path)
    print("%-24s shape=%-10s dtype=%-8s %5d 字节" % (
        name, arr.shape, dt.__name__, os.path.getsize(path)))

NLP 模型的 `input_ids` 常为 int64。dtype 错误时，文件字节数可能仍然匹配，但推理会读取到错误的数值。

#### 读取输出

msame 在 `--output` 下按时间戳创建子目录，输出文件名包含输出节点名称和序号。


In [ ]:
import glob

run_dirs = sorted(glob.glob("l08_workspace/msame_out/*"))
latest = run_dirs[-1]
print("最新一次:", latest)

for f in sorted(glob.glob(latest + "/*")):
    print("  %-40s %6d 字节" % (os.path.basename(f), os.path.getsize(f)))

读取前先核算字节数：实际字节数除以 dtype 字节宽，应等于 shape 各维之积。

#### 比较 ONNX Runtime 与 OM 输出


In [ ]:
out_file = sorted(glob.glob(latest + "/*.bin"))[0]
expect_shape = (1, 10)

n_bytes = os.path.getsize(out_file)
n_elem = n_bytes // 4
print("字节数 %d，float32 元素数 %d，期望 %d" % (n_bytes, n_elem, np.prod(expect_shape)))
assert n_elem == np.prod(expect_shape)

om_out = np.fromfile(out_file, dtype=np.float32).reshape(expect_shape)
print(np.round(om_out[0][:5], 4))

使用同一份输入分别运行 ONNX Runtime 与 OM，比较分类结果、余弦相似度和最大绝对误差。


In [ ]:
import onnxruntime as ort
import numpy as np

x_np = np.fromfile(
    "l08_workspace/input.bin",
    dtype=np.float32
).reshape(1, 3, 32, 32)


# 创建 ORT 配置
sess_options = ort.SessionOptions()

# 关键：显式指定线程数
sess_options.intra_op_num_threads = 1
sess_options.inter_op_num_threads = 1


sess = ort.InferenceSession(
    "l08_workspace/demo_model.onnx",
    sess_options=sess_options,
    providers=["CPUExecutionProvider"]
)


ref = sess.run(None, {"input": x_np})[0]


cos = float(
    np.dot(ref.ravel(), om_out.ravel()) /
    (np.linalg.norm(ref) * np.linalg.norm(om_out))
)


print("① argmax     : ref=%d om=%d  %s" % (
    ref.argmax(),
    om_out.argmax(),
    "一致" if ref.argmax() == om_out.argmax() else "不一致"
))

print("② 余弦相似度 : %.6f" % cos)

print("③ 最大绝对误差: %.3e" % np.abs(ref - om_out).max())

分类任务可先比较 argmax。余弦相似度用于观察整体分布，逐元素误差有助于定位差异。默认 FP16 精度模式下，逐元素误差会随网络层数累计；需要分析精度模式时，可在 08.02 中转换 `must_keep_origin_dtype` 版本作对照。


### 3. 统计推理耗时

首次推理包含模型加载、内存分配和算子初始化，统计稳定态时应单独处理。

| 指标 | 定义 | 用途 |
| --- | --- | --- |
| 首次时延 | 第一次执行的耗时 | 冷启动成本 |
| 平均时延 | (总耗时 − 首次) / 有效次数 | 平均执行开销 |
| 标准差 | 时延的波动 | 观察稳定性 |
| 吞吐量 | 有效次数 × batch / 有效总耗时 | 批处理场景 |

使用 `--loop` 运行多次。预热次数和统计范围可结合本机输出确定。


In [ ]:
# %%bash
# source /usr/local/Ascend/ascend-toolkit/set_env.sh

!/home/ma-user/work/tools/msame/out/msame --model l08_workspace/demo_model.om \
      --input l08_workspace/input.bin \
      --output l08_workspace/perf_out \
      --outfmt BIN \
      --loop 15

从 msame 输出中取得每次耗时，示例将前 5 次作为预热数据：


In [ ]:
# 把 msame 输出的每次耗时填进来（单位 ms）
lat = [12.83, 1.42, 1.38, 1.45, 1.39, 1.41, 1.37, 1.44,
       1.40, 1.38, 1.43, 1.39, 1.41, 1.38, 1.42]

warmup = 5
valid = np.array(lat[warmup:])

print("首次时延      %.2f ms" % lat[0])
print("含首次的平均  %.2f ms" % np.mean(lat))
print("去预热的平均  %.2f ms" % valid.mean())
print("标准差        %.3f ms" % valid.std())
print("P95           %.2f ms" % np.percentile(valid, 95))
print("吞吐          %.1f 次/秒" % (1000.0 / valid.mean()))

记录性能数据时，同时记录硬件型号、CANN 版本、输入规模和循环次数。


### 4. 处理动态 shape

静态场景只需基本参数。动态场景下，msame 还需要知道本次实际 shape 和输出 buffer 上限。

| 参数 | 用途 |
| --- | --- |
| `--dymShape` | 本次推理的实际 shape，格式 `输入名:维度,...` |
| `--outputSize` | 每个输出的字节上限，有几个输出给几个数字 |
| `--dymBatch` | 动态 batch 场景指定档位 |

`--dymShape` 的值应落在 ATC 的 `--input_shape_range` 内。`--outputSize` 按输出 shape 和 dtype 计算：


In [ ]:
outputs = [((1, 10), np.float32), ((1, 128, 768), np.float32)]

sizes = []
for shape, dt in outputs:
    n = int(np.prod(shape)) * np.dtype(dt).itemsize
    sizes.append(int(n * 1.2))          # 留 20% 余量
    print("%-18s %8d 字节 -> 取 %d" % (str(shape), n, sizes[-1]))

print('\n--outputSize "%s"' % ",".join(str(s) for s in sizes))

In [ ]:
# %%bash
# source /usr/local/Ascend/ascend-toolkit/set_env.sh

!/home/ma-user/work/tools/msame/out/msame --model l08_workspace/demo_model_dyn.om \
      --input l08_workspace/input.bin \
      --output l08_workspace/dyn_out \
      --outfmt BIN \
      --dymShape "input:1,3,32,32" \
      --outputSize "48"

### 5. 按阶段定位问题

将问题按环境、加载、参数、数据和结果分类，有助于缩小排查范围。

| 阶段 | 症状 | 常见根因 |
| --- | --- | --- |
| 环境 | 命令找不到、库加载失败 | 未 source `set_env.sh`；msame 未编译 |
| 加载 | 模型加载失败或超时 | `soc_version` 不匹配；OM 文件损坏；模型过大 |
| 参数 | 报参数错误 | `--dymShape` 超出范围；`--outputSize` 缺失或不足 |
| 数据 | 输入尺寸不匹配 | bin 字节数与 shape 不符；dtype 错 |
| 结果 | 数值差异较大 | reshape 错；精度模式；算子实现差异 |

`atc` 或 `msame` 命令找不到时，先检查环境脚本；每个 notebook cell 都是独立进程。`Input shape not fully specified` 表示动态维缺少取值范围或档位配置。输入尺寸错误时，比较 `prod(shape) × dtype 字节宽` 与 bin 文件大小。

下面的代码检查环境变量、OM 文件和 bin 字节数：


In [ ]:
import os, numpy as np

print("ASCEND_TOOLKIT_HOME:", os.environ.get("ASCEND_TOOLKIT_HOME", "未设置"))

om = "l08_workspace/demo_model.om"
print("OM 文件:", os.path.exists(om), os.path.getsize(om) if os.path.exists(om) else "")

bin_path, shape, dt = "l08_workspace/input.bin", (1, 3, 32, 32), np.float32
expect = int(np.prod(shape)) * np.dtype(dt).itemsize
actual = os.path.getsize(bin_path)
print("bin 字节数: 实际 %d，期望 %d，%s" % (
    actual, expect, "匹配" if actual == expect else "不匹配"))

### 6. 处理 ATC 转换失败

OM 未生成时，回到 ATC 日志定位问题。

| 类别 | 典型报错 | 处理方向 |
| --- | --- | --- |
| 环境 | `atc: command not found` | source `set_env.sh` |
| 参数 | `--soc_version` 为空、framework 不符 | 检查命令参数 |
| shape | `Input shape not fully specified` | 固定 shape，或配置动态档位 |
| 算子 | `Not supported operator: xxx` | 检查算子支持情况 |
| 资源 | 转换阶段 OOM | 减小 batch，`--buffer_optimize=optimize_for_memory` |

算子不支持时，可尝试提高 `opset_version` 重新导出，或用基础算子改写等价语义。确有需要时再评估自定义算子。


## 实验总结

```text
model.pth ──export──► model.onnx ──atc──► model.om
                          │                   │
   检查方式          onnx.checker          msame + 读回 bin
                   onnxruntime 对比        输出比较
```

msame 比较的是转换前后单样本前向输出。数据集级精度、长时间稳定性和并发性能需要使用相应的评测方法。


## 实验扩展

1. 为三输入 NLP 模型准备 bin 文件，用本分册的方法验证文件顺序与字节数。
2. 用第 3 节的方法统计一次 15 循环的耗时，给出各项指标并说明首次与预热数据的处理方式。
3. 给定三条报错，指出所属阶段和下一步动作：
   - `E19999: Inner Error, Not supported operator: GridSample`
   - `Expected size: 12288, but given input size: 6144`
   - `terminate called after throwing an instance of 'std::runtime_error'`（msame 启动瞬间）
4. 一个模型有两个 float32 输出，shape 分别为 `(1, 1000)` 和 `(1, 256, 14, 14)`，计算 `--outputSize`。
5. 让输入 bin 少 4 个字节，运行 msame，记录报错并解释。
6. 在昇腾环境运行 `msame --help`，核对本分册使用的参数并记录差异。
7. 为什么随机输入可能掩盖问题？哪些场景应使用真实数据？
8. msame 输出对齐后，业务指标仍可能受哪些因素影响？


## 参考答案


In [ ]:
!cat answer/L08-03_answer.txt